# D07_S01 Refugee Camp for 6 Homes
Design the system for a  120V, AC distribution system for 24 households. You have access to the same 50mx50m square to deploy the PV and batteries and single phase inverters.

The requirement is to supply electricity for an ADMD of 300W, and that the loads are assumed to be a constant power load, with pf =0.95 . The national regulator states that all AC systems must be within 10% of the nominal voltage. After testing of the existing lines, it is found that they have a ratio of X/R = 1. (Use R = 0.37 Ω)

Extend your Gauss Seidel code to model the system to determine the voltage at each node.
1) Determine Pbase Vbase  and Zbase .
2) Use R from the DC case, and convert to PU. Calc X. 
3) Do GS using PU values
4) Revert to actual values if you want to.


In [1]:
# Import numpy
import numpy as np
import pandapower as pp


## PandaPower Solution

In [2]:
# Create an empty network
net = pp.create_empty_network()

system_voltage = 120/1000

# Set the voltages for the buses
bus1 = pp.create_bus(net, vn_kv=system_voltage, name="Bus 1")
bus2 = pp.create_bus(net, vn_kv=system_voltage, name="Bus 2")
bus3 = pp.create_bus(net, vn_kv=system_voltage, name="Bus 3")
bus4 = pp.create_bus(net, vn_kv=system_voltage, name="Bus 4")
bus5 = pp.create_bus(net, vn_kv=system_voltage, name="Bus 5")
bus6 = pp.create_bus(net, vn_kv=system_voltage, name="Bus 6")
bus7 = pp.create_bus(net, vn_kv=system_voltage, name="Bus 7")

# inspect bus objects
net.bus

,name,vn_kv,type,zone,in_service,geo
0,Bus 1,0.12,b,None,True,None
1,Bus 2,0.12,b,None,True,None
2,Bus 3,0.12,b,None,True,None
3,Bus 4,0.12,b,None,True,None
4,Bus 5,0.12,b,None,True,None
5,Bus 6,0.12,b,None,True,None
6,Bus 7,0.12,b,None,True,None


In [3]:
# Define slack bus
slack = pp.create_ext_grid(net, bus=bus1, vm_pu=1, name="Swing bus")
# inspect bus objects
net.bus


,name,vn_kv,type,zone,in_service,geo
0,Bus 1,0.12,b,None,True,None
1,Bus 2,0.12,b,None,True,None
2,Bus 3,0.12,b,None,True,None
3,Bus 4,0.12,b,None,True,None
4,Bus 5,0.12,b,None,True,None
5,Bus 6,0.12,b,None,True,None
6,Bus 7,0.12,b,None,True,None


In [4]:
# add loads
P=300/1e6
pf=0.95
q= P*np.tan(np.arccos(pf))


pp.create_load(net, bus=bus2, p_mw=P, q_mvar=q, name="Load 2")
pp.create_load(net, bus=bus3, p_mw=P, q_mvar=q, name="Load 3")
pp.create_load(net, bus=bus4, p_mw=P, q_mvar=q, name="Load 4")
pp.create_load(net, bus=bus5, p_mw=P, q_mvar=q, name="Load 5")
pp.create_load(net, bus=bus6, p_mw=P, q_mvar=q, name="Load 6")
pp.create_load(net, bus=bus7, p_mw=P, q_mvar=q, name="Load 7")

print(q)

9.860523155365896e-05


In [5]:
# Line data
length_km = 50 / 1000            # each stretch is 50 m = 0.05 km
R_stretch = 0.37                 # ohms for one whole stretch
r_per_km  = R_stretch / length_km   # 7.4 ohm per km, so that 7.4 x 0.05 = 0.37
x_per_km  = r_per_km             # X/R = 1, so X equals R

# Six lines joining the seven buses in a chain
pp.create_line_from_parameters(net, from_bus=bus1, to_bus=bus2, length_km=length_km,
    r_ohm_per_km=r_per_km, x_ohm_per_km=x_per_km, c_nf_per_km=0, max_i_ka=0.1, name="Line 1-2")
pp.create_line_from_parameters(net, from_bus=bus2, to_bus=bus3, length_km=length_km,
    r_ohm_per_km=r_per_km, x_ohm_per_km=x_per_km, c_nf_per_km=0, max_i_ka=0.1, name="Line 2-3")
pp.create_line_from_parameters(net, from_bus=bus3, to_bus=bus4, length_km=length_km,
    r_ohm_per_km=r_per_km, x_ohm_per_km=x_per_km, c_nf_per_km=0, max_i_ka=0.1, name="Line 3-4")
pp.create_line_from_parameters(net, from_bus=bus4, to_bus=bus5, length_km=length_km,
    r_ohm_per_km=r_per_km, x_ohm_per_km=x_per_km, c_nf_per_km=0, max_i_ka=0.1, name="Line 4-5")
pp.create_line_from_parameters(net, from_bus=bus5, to_bus=bus6, length_km=length_km,
    r_ohm_per_km=r_per_km, x_ohm_per_km=x_per_km, c_nf_per_km=0, max_i_ka=0.1, name="Line 5-6")
pp.create_line_from_parameters(net, from_bus=bus6, to_bus=bus7, length_km=length_km,
    r_ohm_per_km=r_per_km, x_ohm_per_km=x_per_km, c_nf_per_km=0, max_i_ka=0.1, name="Line 6-7")

# inspect the line objects
net.line

,name,std_type,from_bus,to_bus,length_km,r_ohm_per_km,x_ohm_per_km,c_nf_per_km,g_us_per_km,max_i_ka,df,parallel,type,in_service,geo
0,Line 1-2,None,0,1,0.05,7.4,7.4,0.0,0.0,0.1,1.0,1,None,True,None
1,Line 2-3,None,1,2,0.05,7.4,7.4,0.0,0.0,0.1,1.0,1,None,True,None
2,Line 3-4,None,2,3,0.05,7.4,7.4,0.0,0.0,0.1,1.0,1,None,True,None
3,Line 4-5,None,3,4,0.05,7.4,7.4,0.0,0.0,0.1,1.0,1,None,True,None
4,Line 5-6,None,4,5,0.05,7.4,7.4,0.0,0.0,0.1,1.0,1,None,True,None
5,Line 6-7,None,5,6,0.05,7.4,7.4,0.0,0.0,0.1,1.0,1,None,True,None


In [6]:
# Solve the power flow (this replaces the whole Gauss-Seidel while loop)
pp.runpp(net, algorithm="gs")

# Voltages at every bus
net.res_bus

,vm_pu,va_degree,p_mw,q_mvar
0,1.000000,0.000000,-0.002205,-0.000997
1,0.918239,-1.937684,0.000300,0.000099
2,0.848672,-3.840332,0.000300,0.000099
3,0.792054,-5.604899,0.000300,0.000099
4,0.749026,-7.104321,0.000300,0.000099
5,0.720074,-8.203805,0.000300,0.000099
6,0.705520,-8.787426,0.000300,0.000099


## Set the coordinates

In [7]:
for i, v in enumerate([V1_k, V2_k, V3_k, V4_k, V5_k, V6_k], start=1):
    print(f"V{i} = {np.abs(v):.4f} ∠ {np.angle(v, deg=True):.2f}°")

NameError: name 'V1_k' is not defined

## Determine all the currents and power flows

In [16]:
I01 = Y * (V0 - V1_k)        # current from hub to Home 1 (already in the template)
S0  = V0 * np.conj(I01)      # power the hub sends out

print(f"S0 = {S0 * SBase:.4f} ")

S0 = 2103.2529+894.9149j 


In [9]:
# currents
I12 = Y * (V1_k - V2_k);  I21 = -I12
I23 = Y * (V2_k - V3_k);  I32 = -I23
I34 = Y * (V3_k - V4_k);  I43 = -I34
I45 = Y * (V4_k - V5_k);  I54 = -I45
I56 = Y * (V5_k - V6_k);  I65 = -I56


In [10]:
# power
IBase = SBase / VBase
for name, I in [("I01", I01), ("I12", I12), ("I23", I23), ("I34", I34), ("I45", I45), ("I56", I56)]:
    print(f"{name} = {np.abs(I)*IBase:.2f} A")

I01 = 20.17 A
I12 = 17.31 A
I23 = 14.22 A
I34 = 10.90 A
I45 = 7.38 A
I56 = 3.73 A


## Max ADMD
If the supply voltage is increased to a nominal voltage of 230V, what is the maximum ADMD that can be supplied using the original DC lines, whilst still meeting the Voltage requirements.


In [11]:
def v6_for(P, VBase=230, SBase=1000, R=0.37, XR=1, pf=0.95):
    ZBase = VBase**2 / SBase
    Rpu = R / ZBase
    Y = 1 / (Rpu + 1j*Rpu*XR)
    Ph = P / SBase
    Qh = Ph * np.tan(np.arccos(pf))
    Sh_conj = -Ph + 1j*Qh
    V0 = 1+0j
    V = [1+0j]*6                      # V[0] is Home 1 ... V[5] is Home 6
    e, count = 1.0, 0
    while e > 1e-8 and count < 5000:
        old = V.copy()
        V[0] = (Sh_conj/np.conj(V[0]) + Y*V0   + Y*V[1]) / (2*Y)
        V[1] = (Sh_conj/np.conj(V[1]) + Y*V[0] + Y*V[2]) / (2*Y)
        V[2] = (Sh_conj/np.conj(V[2]) + Y*V[1] + Y*V[3]) / (2*Y)
        V[3] = (Sh_conj/np.conj(V[3]) + Y*V[2] + Y*V[4]) / (2*Y)
        V[4] = (Sh_conj/np.conj(V[4]) + Y*V[3] + Y*V[5]) / (2*Y)
        V[5] = (Sh_conj/np.conj(V[5]) + Y*V[4])          / Y
        e = max(abs(V[i] - old[i]) for i in range(6))
        count += 1
    return np.abs(V[5])

P = 300
while v6_for(P + 1) >= 0.9:
    P += 1
print(f"Max ADMD = {P} W, V6 = {v6_for(P):.4f} pu")

Max ADMD = 465 W, V6 = 0.9000 pu
